In [ ]:
import warnings
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")


In [ ]:
# Parameters (dapat dioverride via Papermill dengan flag -p)
data_path = 'artifact/data/BBCA.csv'
train_size_ratio = 0.9
target_col = 'open'


In [ ]:
# Load data secara dinamis berdasarkan data_path yang diinjeksi
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Identifikasi simbol saham
if 'symbol' in df.columns and len(df) > 0:
    symbol = str(df['symbol'].iloc[0])
else:
    symbol = Path(data_path).stem
print(f"Memproses data saham: {symbol} dari {data_path} (Total baris: {len(df)})")


## Data Understanding

### Pengecekan Informasi Data

In [ ]:
start_date = df['date'].min()
end_date = df['date'].max()
total_days = (end_date - start_date).days

print(f"Tanggal Awal Data  : {start_date.strftime('%Y-%m-%d')}")
print(f"Tanggal Akhir Data : {end_date.strftime('%Y-%m-%d')}")
print(f"Total Rentang Waktu: {total_days} Hari Kalender")
print(f"Total Hari Perdagangan Aktif: {len(df)} Hari Kerja")

In [ ]:
df.head()

### Pengecekan Data Hilang

In [ ]:
print(df.isna().sum())

### Visualisasi Harga

In [ ]:
df_idx = df.set_index('date')

plt.figure(figsize=(12, 6))
if 'open' in df_idx.columns and 'close' in df_idx.columns:
    plt.plot(df_idx['open'], label='Open', color='blue', alpha=0.7)
    plt.plot(df_idx['close'], label='Close', color='orange', alpha=0.7)
    plt.title(f'Tren Pergerakan Harga Saham {symbol} (Open vs Close)')
else:
    plt.plot(df_idx[target_col], label=target_col.capitalize(), color='blue', alpha=0.7)
    plt.title(f'Tren Pergerakan Harga Saham {symbol} ({target_col.capitalize()})')
plt.xlabel('Tahun')
plt.ylabel('Harga (Rupiah)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## Data Preparation

### Pembagian Data Latih dan Uji

In [ ]:
TRAIN_SIZE_RATIO = train_size_ratio

train_size = int(len(df) * TRAIN_SIZE_RATIO)
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

print(f"Jumlah Data Train : {len(train_df)}")
print(f"Jumlah Data Test  : {len(test_df)}")


### Dekomposisi Seasonal

In [ ]:
decomposition = seasonal_decompose(df_idx[target_col], model='additive', period=5)

fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.suptitle(f'Dekomposisi Seasonal Harga Saham {symbol}', y=1.02)
plt.show()


## Modelling & Evaluation

In [ ]:
train_y = train_df[target_col].values
test_y = test_df[target_col].values


### Model ARIMA

In [ ]:
p = d = q = range(0, 3)
pdq = list(itertools.product(p, d, q))

best_arima_mape = float("inf")
best_arima_order = None
arima_results = []

In [ ]:
for param in pdq:
    try:
        model = ARIMA(train_y, order=param)
        res = model.fit()
        forecast = res.forecast(steps=len(test_y))

        mse = mean_squared_error(test_y, forecast)
        mape = np.mean(np.abs((test_y - forecast) / test_y)) * 100

        arima_results.append({'order': param, 'MSE': mse, 'MAPE': mape})

        if mape < best_arima_mape:
            best_arima_mape = mape
            best_arima_order = param
    except Exception:
        continue

In [ ]:
arima_results_df = pd.DataFrame(arima_results).sort_values(by='MAPE')
arima_results_df.head()

### Model SARIMA

In [ ]:
p = d = q = range(0, 3)
P = D = Q = range(0, 3)
s = 5

pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], s) for x in list(itertools.product(P, D, Q))]

best_sarima_mape = float("inf")
best_sarima_order = None
best_sarima_seasonal = None
sarima_results = []

In [ ]:
for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            model = sm.tsa.statespace.SARIMAX(
                train_y,
                order=param,
                seasonal_order=param_seasonal,
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            res = model.fit(disp=False)
            forecast = res.forecast(steps=len(test_y))

            mse = mean_squared_error(test_y, forecast)
            mape = np.mean(np.abs((test_y - forecast) / test_y)) * 100

            sarima_results.append({
                'order': param,
                'seasonal_order': param_seasonal,
                'MSE': mse,
                'MAPE': mape
            })

            if mape < best_sarima_mape:
                best_sarima_mape = mape
                best_sarima_order = param
                best_sarima_seasonal = param_seasonal
        except Exception:
            continue

In [ ]:
sarima_results_df = pd.DataFrame(sarima_results).sort_values(by='MAPE')
sarima_results_df.head()

### Visualisasi Hasil

In [ ]:
model_arima = ARIMA(train_y, order=best_arima_order)
res_arima = model_arima.fit()

forecast_arima = res_arima.forecast(steps=len(test_y))

plt.figure(figsize=(10, 5))
plt.plot(train_df['date'], train_df[target_col], color='blue', label='Train Data')
plt.plot(test_df['date'], test_df[target_col], color='green', label='Test Data')
plt.plot(test_df['date'], forecast_arima, color='yellow', label='Forecast')
plt.title(f'Forecast ARIMA {best_arima_order} - {symbol}')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


In [ ]:
model_sarima = sm.tsa.statespace.SARIMAX(
    train_y,
    order=best_sarima_order,
    seasonal_order=best_sarima_seasonal
)
res_sarima = model_sarima.fit(disp=False)

forecast_sarima = res_sarima.forecast(steps=len(test_y))

plt.figure(figsize=(10, 5))
plt.plot(train_df['date'], train_df[target_col], color='blue', label='Train Data')
plt.plot(test_df['date'], test_df[target_col], color='green', label='Test Data')
plt.plot(test_df['date'], forecast_sarima, color='yellow', label='Forecast')
plt.title(f'Forecast SARIMA {best_sarima_order}x{best_sarima_seasonal} - {symbol}')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
